In [ ]:
# =========================================
# DESeq2 for pseudo-replicate pseudo-bulk
# =========================================

suppressPackageStartupMessages({
library(DESeq2)
library(tidyverse)
library(apeglm)
library(here)
})

In [ ]:
setwd('..')

In [ ]:
getwd()

In [ ]:
utils <- new.env()

sys.source(here::here("scripts", "utils.r"), envir = utils)

In [ ]:
counts <- read_csv('./CSV/pb_counts.csv') %>%  as.data.frame()
coldata <- read_csv("./CSV/pb_meta.csv") %>%  as.data.frame()

In [ ]:
# Setting rownames
rownames(counts) <- counts[[1]]; counts[[1]] <- NULL
rownames(coldata) <- coldata[[1]]; coldata[[1]] <- NULL

In [ ]:
dds <- DESeqDataSetFromMatrix(countData = counts,
                              colData = coldata,
                              design = ~ infection_group)

In [ ]:
dds <- DESeq(dds)

In [ ]:
dds

In [ ]:
resultsNames(dds)

In [ ]:
utils$analyze_save(dds, "infection_group", "Low infection", "No infection", './Pseudobulk/Sig_deg/', lfc_cutoff = 0.58)
utils$analyze_save(dds, "infection_group", "High infection", "No infection", './Pseudobulk/Sig_deg/', lfc_cutoff = 0.58)
utils$analyze_save(dds, "infection_group", "High infection", "Low infection", './Pseudobulk/Sig_deg/', lfc_cutoff = 0.58)

In [ ]:
suppressPackageStartupMessages({
  library(matrixStats) 
})

# ===== PCA (DESeq2 vst based) =====
vsd <- vst(dds, blind = TRUE)
mat <- assay(vsd)

pca <- prcomp(t(mat), center = TRUE, scale. = FALSE)

pca_df <- as.data.frame(pca$x)
pca_df$group  <- colData(dds)$condition
pca_df$sample <- colnames(mat)

var_ratio <- (pca$sdev ^ 2) / sum(pca$sdev ^ 2)
pc1_var   <- round(var_ratio[1] * 100, 1)
pc2_var   <- round(var_ratio[2] * 100, 1)

vr <- data.frame(PC = paste0("PC", seq_along(var_ratio)),
                 ratio = var_ratio)

In [ ]:
write.csv(pca_df, './Pseudobulk/pca.csv', row.names = FALSE)
write.csv(vr, "./Pseudobulk/variance.csv", row.names = FALSE)

In [ ]:
sessionInfo()